# Beaver Simulation - Boston Rumney Marsh

This notebook runs a multi-agent beaver simulation on real DEM (Digital Elevation Model) data.

## Quick Start Guide

1. **Run Cell 1**: Setup Python path
2. **Configure Cell 2**: Adjust parameters (see guide below)
3. **Run Cell 3**: Start simulation

## Parameter Guide

### Environment Parameters
- **File paths**: Point to DEM data (elevation, X/Y coordinates)
- **vegetation_quality_range**: `[min, max]` vegetation values (usually `[0, 10]`)
- **grass_growth_rate**: How fast vegetation grows (e.g., `1e-4` = 0.01% per hour)
- **grass_growth_interval**: `[min, max]` elevation where grass grows
- **streams_width**: Depth below 0 for water bodies (negative values = water)
- **river_growth_velocity**: How fast rivers deepen over time

### Robot (Beaver) Parameters
- **position**: `'random_home'` (near home), `'home'` (at home), or `[x, y]` coordinates
- **home_base_position**: `[[x, y]]` where beaver lodge is located (check `processing_metadata.json`)
- **range_x/y**: Random offset from home when using `'random_home'` (e.g., `[-3, 3]`)
- **maximum_load**: Max vegetation beaver can carry before returning home
- **harvest_threshold**: `[min, max]` vegetation quality to harvest (e.g., `[5, 10]`)
- **vegetation_removal**: Amount removed per harvest action
- **exploration_mode**: How beavers explore (e.g., `'gradient_D010'` = 10-cell neighborhood)
- **exploration_eta**: Exploration randomness (0=deterministic, 1=random)
- **exploration_map**: `'vegetation_quality'` or `'vegetation_visits'` guides exploration
- **n_traces**: Number of recent steps to remember for adaptive behavior
- **decay_values**: `[eta, threshold, load]` decay rates for adaptive learning

### Controller Parameters (Movement)
- **name**: `'P_repulsive'` (with terrain avoidance) or `'P'` (simple)
- **Kp, Kd, Ki**: PID controller gains (proportional, derivative, integral)
- **neighbourhood_size**: Cells around beaver to consider for repulsion
- **map_repulsive**: Map used for terrain avoidance

### Simulation Parameters
- **number_of_agents**: How many beavers to simulate
- **number_of_steps**: Total simulation time (in hours, e.g., `365*2` = 2 years)
- **downsampling**: Save frequency (e.g., `7*24` = weekly snapshots)
- **timedelta**: Time step size (1.0 = 1 hour per step)

---

In [1]:
# Setup: Add parent directory to Python path to import beaversim modules
import sys
import os
from pathlib import Path

# Get the notebook's directory and add the parent (project root) to sys.path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Notebook directory: {notebook_dir}")
print(f"Project root: {project_root}")
print(f"Python path updated to include: {project_root}")

Notebook directory: /home/fedeoli/Documents/Work/Projects/beavers/beaverbot/notebooks
Project root: /home/fedeoli/Documents/Work/Projects/beavers/beaverbot
Python path updated to include: /home/fedeoli/Documents/Work/Projects/beavers/beaverbot


In [2]:
from beaversim.scenarios.standard_beavers_scenario import standard_beavers_scenario

# ============================================================================
# BEAVER SIMULATION CONFIGURATION - RUMNEY MARSH, BOSTON
# ============================================================================

beavers_running_config = {
    'backend_type': 'beavers_visualizer',
    
    # ========================================================================
    # ENVIRONMENT CONFIGURATION
    # ========================================================================
    'environment': {
        'name': 'beavers_environment',
        
        # --- Data Files (DEM outputs from utils/dem_conversion_interactive.ipynb) ---
        'elevation_file_path': '../beaversim/utils/output/RumneyMarsh_Boston/elevation.npy',
        'latitude_file_path': '../beaversim/utils/output/RumneyMarsh_Boston/X_coordinates.npy',
        'longitude_file_path': '../beaversim/utils/output/RumneyMarsh_Boston/Y_coordinates.npy',
        
        # --- Vegetation Dynamics ---
        'vegetation_quality_range': [0, 10],      # Min/max vegetation values
        'grass_growth_rate': 1e-4,                # Growth rate per hour (0.01% per hour)
        'grass_growth_interval': [0, 4],          # Elevation range where grass grows
        'visits_reset': 1,                        # (Internal parameter)
        
        # --- Water/River Dynamics ---
        'flow_info': {
            'direction': [0, 0],                  # Flow direction vector [x, y]
            'strength': 0.0,                      # Flow strength (affects beaver movement)
            'streams_width': 10,                  # Water depth (elevation < 0 = water)
            'river_growth_velocity': 1e-4,        # Rate rivers deepen over time
            'river_growth_interval': [-3, 0],     # Elevation range where rivers deepen
        },
        
        'print': False,                           # Debug printing
    },
    
    # ========================================================================
    # BEAVER (ROBOT) CONFIGURATION
    # ========================================================================
    'robot': {
        'name': 'beavers_robot',
        
        # --- Initial Position ---
        # Options: 'random_home' (near home base), 'home' (at home base), [x,y] coordinates
        'position': 'random_home',
        'home_base_position': [[150, 550]],       # Lodge location (see processing_metadata.json)
        'range_x': [-3, 3],                       # Random spawn offset from home (x-axis)
        'range_y': [-3, 3],                       # Random spawn offset from home (y-axis)
        
        # --- Capacity & Energy ---
        'maximum_load': 1000,                     # Max load before returning home
        
        # --- Harvesting Behavior ---
        'harvest_threshold': [3, 8],             # Only harvest vegetation in this range
        'vegetation_removal': 1.0,                # Amount removed per harvest action
        
        # --- Perception & Exploration ---
        'measurement_mode': 'full_map',           # 'full_map' = omniscient, 'local' = limited view
        'exploration_mode': 'gradient_D010',      # Gradient-based with 10-cell neighborhood
        'measure_step': 1,                        # Measurement grid step size
        'exploration_eta': [0.5],                 # Exploration randomness (0-1, higher = more random)
        'exploration_map': 'vegetation_visits',   # Guide exploration by: 'vegetation_quality' or 'vegetation_visits'
        'n_traces': 30,                           # Memory: recent steps to track for adaptation
        'decay_values': [0, 0, 0],                # Adaptive decay [eta, harvest_threshold, max_load]
        'exploration_N_recovery': 10,             # Steps before re-exploring same area
        
        # --- Movement Controller (PID) ---
        'controller': {
            'name': 'P_repulsive',                # 'P_repulsive' includes terrain avoidance
            'Kp': 0.0,                            # Proportional gain
            'Kd': 0.0,                            # Derivative gain
            'Ki': 0.005,                          # Integral gain (primary for smooth movement)
            'neighbourhood_size': 2,              # Cells around beaver for repulsion
            'max': 2.0,                           # Max movement speed
            'accuracy': 1e-2,                     # Position accuracy threshold
            'dimension': 2,                       # 2D space
            'map_repulsive': 'vegetation_quality',  # Map for terrain avoidance
            'beta_repulsive': 0.5,                # Repulsion strength
        },
        
        # --- Physics Model ---
        'dynamics': {
            'name': 'integrator',                 # Simple integrator dynamics
            'mass': 1.0,
            'friction': 1.0,
        },
        
        # --- Agent Role (Multi-Agent) ---
        'role': 'explorer',                       # 'explorer' seeks vegetation, 'builder' prefers low quality
        
        'print': False,                           # Debug printing
    },
    
    # ========================================================================
    # SIMULATION CONFIGURATION
    # ========================================================================
    'simulation': {
        'seed': 10,                               # Random seed for reproducibility
        'timedelta': 1.0,                         # Time step size (hours)
        'schedule_policy': 'sequential',          # Agent execution order
        'gui': True,                              # Enable visualization
        'number_of_agents': 10,                   # Number of beavers
        'number_of_steps': 365 * 2,               # Total steps (2 years in hours)
        'downsampling': 7 * 24,                   # Save every 7 days (weekly)
        'save_path': 'output/environment_map_boston_rumney_marsh.npy',
        'print': True,                            # Print simulation progress
    },
}

In [3]:
standard_beavers_scenario(beavers_running_config)

ValueError: Error loading NPY file: [Errno 2] No such file or directory: '../beaversim/utils/output/RumneyMarsh_Boston/elevation.npy'